In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


# Group 1

In [2]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
train_dataset = train_dataset_full
test_dataset = test_dataset_full

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

train - test dataset ready.


# Group 2

In [4]:
# --- Part 2: Configure Model ---
# Task 2: Load pretrained ResNet18
print("\nLoading pretrained ResNet18 model...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Task 3: Modify the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
# Task 4: Freeze all layers except layer3, layer4, and fc
# First, freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the layers for fine-tuning
print("Unfreezing layers for fine-tuning: layer3, layer4, and fc.")
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

#Verification(rishab likes goth chicks)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nModel setup is complete.")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print("\nNotebook is now ready for Group 3 (Fine-Tuning).")


Loading pretrained ResNet18 model...
Unfreezing layers for fine-tuning: layer3, layer4, and fc.

Model setup is complete.
  Total parameters:     11,181,642
  Trainable parameters: 10,498,570

Notebook is now ready for Group 3 (Fine-Tuning).


In [5]:
# --- Group 3: Fine-Tuning, Evaluation, and Feature Extraction ---
model = model.to(device)

# Define loss and optimizer BEFORE training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


# Check for existing saved model
if os.path.exists("full_model.pth"):
    print("Found existing 'full_model.pth' — skipping training and loading saved weights.")
    model.load_state_dict(torch.load("full_model.pth", map_location=device))
    model.eval()

else:
    print("No saved model found — starting training from scratch.")
    
    # --- Early Stopping Setup ---
    patience = 4
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    counter = 0
    num_epochs = 20

    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")
        print("-" * 30)
        
        # --- Training Phase ---
        model.train()
        running_loss, running_corrects, total = 0.0, 0, 0
        
        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = running_corrects.double() / total

        # --- Validation Phase ---
        model.eval()
        val_loss, val_corrects, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(test_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                val_total += labels.size(0)

        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        
        scheduler.step()

        # --- Early Stopping Check ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            counter = 0
            print("Validation accuracy improved — saving best model.")
            torch.save(best_model_wts, "best_model.pth")
        else:
            counter += 1
            print(f"No improvement for {counter} epoch(s).")
            if counter >= patience:
                print("Early stopping triggered.")
                break

    # --- Load best weights after training ---
    model.load_state_dict(best_model_wts)
    print("\nTraining complete. Best Validation Accuracy: {:.4f}".format(best_val_acc))

    # --- Save the final best model ---
    torch.save(model.state_dict(), "full_model.pth")
    print("Best model saved as 'full_model.pth'")


No saved model found — starting training from scratch.

Epoch [1/20]
------------------------------


Training:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 1.0840 | Train Acc: 0.6243
Val   Loss: 0.7884 | Val   Acc: 0.7250
Validation accuracy improved — saving best model.

Epoch [2/20]
------------------------------


Training:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

... (21 outputs left out)

Train Loss: 0.0669 | Train Acc: 0.9778
Val   Loss: 0.9849 | Val   Acc: 0.7753
No improvement for 3 epoch(s).

Epoch [10/20]
------------------------------


Training:   0%|          | 0/782 [00:00<?, ?it/s]

Validating:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 0.0532 | Train Acc: 0.9829
Val   Loss: 1.0591 | Val   Acc: 0.7718
No improvement for 4 epoch(s).
Early stopping triggered.

Training complete. Best Validation Accuracy: 0.7771
Best model saved as 'full_model.pth'


**Training Summary of the saved model:**

* **Total epochs run:** 6
* **Early stopping patience:** 4
* **Best epoch:** 2
* **Best Validation Accuracy:** 0.7771
* **Training Accuracy (at best epoch):** 0.9829
* **Validation Loss (at best epoch):** 1.0591
